# DataBridge feature tour
One notebook that touches **every notebook feature**. Click **Run all** (top right) — it finishes in about a minute on a laptop.

| # | Section | Shows |
|---|---|---|
| 1 | Widgets | `dbutils.widgets` text, dropdown, combobox, multiselect |
| 2 | Create data | PySpark + a Delta (or Parquet) table, **Spark Jobs** progress |
| 3 | `display()` | Table (filters, sort, resize), Chart, **EDA**, full screen |
| 4 | `%sql` | SQL cells, `${widget}` parameters, `_sqldf` |
| 5 | Shared code | `%run ./_helpers` |
| 6 | Child notebooks | `dbutils.notebook.run`, `runMultiple`, `exit` |
| 7 | Delta Lake | history, UPDATE, time travel, OPTIMIZE / Z-ORDER |
| 8 | pandas | `pdf.display()` |
| 9 | HTML | `displayHTML()` |
| 10 | Files | `dbutils.fs`, `%fs` |
| 11 | Shell | `%sh` |
| 12 | Dashboard data | a gold table for **Dashboards › Build with AI** |
| 13 | Notebook tools | TOC, variables, history, shortcuts, Git, Schedule |
| 14 | Errors | **Fix with Dora** (last cell fails on purpose) |

> Tip: the **≡** icon in the header opens the table of contents built from these headings.

## 1. Widgets
Widgets appear in a bar above the cells. Change a value and re-run the cells that use it. In **Run all**, the current values are used.

In [ ]:
dbutils.widgets.dropdown("region", "All", ["All", "North", "South", "East", "West"], "Region")
dbutils.widgets.multiselect("channels", "Online,Store", ["Online", "Store", "Phone"], "Channels")
dbutils.widgets.text("min_order", "100", "Minimum order (₹)")
dbutils.widgets.combobox("target", "2000000", ["1000000", "2000000", "5000000"], "Monthly target")

region = dbutils.widgets.get("region")
channels = dbutils.widgets.get("channels").split(",")
min_order = float(dbutils.widgets.get("min_order"))
print("widgets:", dbutils.widgets.getAll())

## 2. Create sample data
6,000 orders across 4 regions, 3 channels and 5 categories. Watch the **(N) Spark Jobs** bar under the cell: expand it for jobs → stages → tasks, and use **Spark UI ↗** to open the Spark UI.

The table is created as **Delta** if Delta Lake is available, otherwise as Parquet (the Delta sections then explain what they would do).

In [ ]:
from pyspark.sql import functions as F

spark.sql("CREATE DATABASE IF NOT EXISTS demo")
orders = (spark.range(0, 6000, numPartitions=8)
    .withColumn("order_date", F.expr("date_add(DATE'2026-06-01', CAST(id % 120 AS INT))"))
    .withColumn("region", F.expr("element_at(array('North','South','East','West'), CAST(id % 4 AS INT) + 1)"))
    .withColumn("channel", F.expr("element_at(array('Online','Store','Phone'), CAST((id / 4) % 3 AS INT) + 1)"))
    .withColumn("category", F.expr("element_at(array('Grocery','Dairy','Snacks','Beverages','Household'), CAST((id * 7) % 5 AS INT) + 1)"))
    .withColumn("customer_id", (F.col("id") % 900 + 1).cast("int"))
    .withColumn("quantity", (F.col("id") % 5 + 1).cast("int"))
    .withColumn("unit_price", F.round(40 + (F.col("id") * 37) % 460, 2))
    .withColumn("discount", F.round(F.when(F.col("id") % 6 == 0, F.col("unit_price") * 0.1).otherwise(0), 2))
    .withColumn("net_sales", F.round(F.col("quantity") * F.col("unit_price") - F.col("discount"), 2))
    .withColumn("cost", F.round(F.col("net_sales") * 0.68, 2))
    .withColumnRenamed("id", "order_id"))

try:
    orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("demo.sales")
    TABLE_FORMAT = "delta"
except Exception as e:                       # no Delta Lake on this cluster: use Parquet
    spark.sql("DROP TABLE IF EXISTS demo.sales")
    orders.write.format("parquet").mode("overwrite").saveAsTable("demo.sales")
    TABLE_FORMAT = "parquet"
IS_DELTA = TABLE_FORMAT == "delta"
print(f"demo.sales created as {TABLE_FORMAT.upper()} with {spark.table('demo.sales').count():,} rows")

## 3. `display()` — Table, Chart and EDA
Try these on the output below:
- **Table**: hover a header and click the **filter** icon (Values or Condition), click a header to **sort**, drag a column edge to **resize**, use **Search** in the tab bar, download CSV.
- **Chart**: pick an analysis — Histogram, Box plot, Count, Pie, Bar, Line, Scatter (with correlation), Box by group, Heatmap; choose several **Y values** from the dropdown.
- **EDA**: Overview, Columns (click one for its distribution), Relationships (two columns + correlation matrix), Data quality.
- **⤢** (top right of the result) opens it full screen.

In [ ]:
sales = spark.table("demo.sales")
display(sales.orderBy("order_date"))

### Aggregations show the Spark Jobs card too
A `groupBy` causes a **shuffle**, so you'll see more than one stage (and *skipped* stages when Spark reuses work).

In [ ]:
by_region = (sales.where(F.col("net_sales") >= min_order)
    .groupBy("region", "channel")
    .agg(F.round(F.sum("net_sales"), 2).alias("net_sales"), F.count("*").alias("orders"),
         F.round(F.avg("net_sales"), 2).alias("avg_order"))
    .orderBy(F.desc("net_sales")))
by_region.display()

## 4. `%sql` cells
SQL cells use `${widget}` parameters — change **Region** in the widget bar and re-run. The last result is also available in Python as `_sqldf`.

In [ ]:
%sql
SELECT category,
       ROUND(SUM(net_sales), 2)            AS net_sales,
       COUNT(*)                            AS orders,
       ROUND(SUM(net_sales - cost), 2)     AS gross_margin
FROM demo.sales
WHERE ('${region}' = 'All' OR region = '${region}')
  AND net_sales >= ${min_order}
GROUP BY category
ORDER BY net_sales DESC

In [ ]:
top = _sqldf.first()
print(f"Best category: {top['category']} with net sales {top['net_sales']:,.2f}")

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW monthly AS
SELECT date_trunc('month', order_date) AS month, region, ROUND(SUM(net_sales), 2) AS net_sales
FROM demo.sales GROUP BY 1, 2;

SELECT * FROM monthly ORDER BY month, region

## 5. Shared code with `%run`
`%run` executes another notebook **in this notebook's scope**, so its functions and variables become available here (a live card shows its progress).

In [ ]:
%run ./_helpers

In [ ]:
with_margin = add_margin(sales)
total = with_margin.agg(F.sum("net_sales").alias("s"), F.sum("gross_margin").alias("m")).first()
print("Net sales:", inr(total["s"]), "| Gross margin:", inr(total["m"]))

## 6. Child notebooks: `dbutils.notebook.run` and `runMultiple`
Each child runs **in its own session** with arguments (read there as widgets) and returns a value with `dbutils.notebook.exit()`. A live card shows each run; finished runs are listed under **Workflows › Notebook runs**.

In [ ]:
import json
result = json.loads(dbutils.notebook.run("./child_region_report", 300, {"region": "West", "table": "demo.sales"}))
print("West:", inr(result["net_sales"]), "from", result["orders"], "orders")

In [ ]:
runs = [{"path": "./child_region_report", "timeout_seconds": 300, "arguments": {"region": r, "table": "demo.sales"}}
        for r in ["North", "South", "East"]]
results = [json.loads(x) for x in dbutils.notebook.runMultiple(runs, max_parallel=3)]
for r in results:
    print(f"{r['region']:6} {inr(r['net_sales']):>12}  {r['orders']} orders")

## 7. Delta Lake: history, UPDATE, time travel, OPTIMIZE
Also see **Catalog › demo › sales › History & time travel** and **Maintenance** for the same tools in the UI.

In [ ]:
if IS_DELTA:
    spark.sql("UPDATE demo.sales SET discount = discount + 5 WHERE region = 'North' AND channel = 'Phone'")
    display(spark.sql("DESCRIBE HISTORY demo.sales").select("version", "timestamp", "operation", "operationMetrics"))
else:
    print("demo.sales is Parquet here — with Delta Lake this cell shows the table history after an UPDATE.")

In [ ]:
if IS_DELTA:
    v0 = spark.sql("SELECT ROUND(SUM(discount), 2) AS d FROM demo.sales VERSION AS OF 0").first()["d"]
    now = spark.sql("SELECT ROUND(SUM(discount), 2) AS d FROM demo.sales").first()["d"]
    print(f"Total discount — version 0: {v0:,.2f} | now: {now:,.2f} (the UPDATE added {now - v0:,.2f})")
else:
    print("Time travel (VERSION AS OF / TIMESTAMP AS OF) needs Delta Lake.")

In [ ]:
if IS_DELTA:
    display(spark.sql("OPTIMIZE demo.sales ZORDER BY (region, order_date)"))
else:
    print("OPTIMIZE / ZORDER compacts small files on Delta tables.")

## 8. pandas
Small results can be converted to pandas — `display()` and `.display()` work on pandas DataFrames too.

In [ ]:
pdf = by_region.toPandas()
pdf["share_pct"] = (pdf["net_sales"] / pdf["net_sales"].sum() * 100).round(1)
pdf.display()

## 9. `displayHTML()`
Render any HTML — handy for a small report card.

In [ ]:
best = pdf.sort_values("net_sales", ascending=False).iloc[0]
displayHTML(f"""
<div style="font-family:system-ui;padding:14px 18px;border-radius:12px;background:linear-gradient(120deg,#0A5CFF,#12CFF5);color:#fff;max-width:520px">
  <div style="opacity:.85;font-size:13px">Best region × channel</div>
  <div style="font-size:26px;font-weight:800">{best['region']} · {best['channel']}</div>
  <div>{inr(best['net_sales'])} from {int(best['orders'])} orders ({best['share_pct']}% of sales)</div>
</div>""")

## 10. Files: `dbutils.fs` and `%fs`
Paths are relative to the workspace.

In [ ]:
dbutils.fs.mkdirs("tmp/feature_tour")
dbutils.fs.put("tmp/feature_tour/notes.txt", "Created by the feature tour\nregion widget = " + region, overwrite=True)
for f in dbutils.fs.ls("tmp/feature_tour"):
    print(f.name, f.size, "bytes")
print(dbutils.fs.head("tmp/feature_tour/notes.txt"))

In [ ]:
%fs ls tmp/feature_tour

## 11. Shell with `%sh`

In [ ]:
%sh
echo "Hello from the shell"
python --version

## 12. Data for Dashboards
This saves a small **gold** table. Then open **Dashboards › Build with AI**, pick `demo.sales_daily`, click **Let AI decide**, and review the proposed dashboard.

In [ ]:
daily = (sales.groupBy("order_date", "region", "channel", "category")
    .agg(F.round(F.sum("net_sales"), 2).alias("net_sales"), F.count("*").alias("orders"),
         F.round(F.sum("discount"), 2).alias("discount"), F.round(F.sum("cost"), 2).alias("cost")))
daily.write.format(TABLE_FORMAT).mode("overwrite").saveAsTable("demo.sales_daily")
print(f"demo.sales_daily: {spark.table('demo.sales_daily').count():,} rows — ready for Build with AI")

## 13. Notebook tools (header, top right)
- **Run all**, ■ interrupt, ↻ restart kernel, ⌫ clear outputs; the status pill shows the kernel state.
- **Schedule** turns this notebook into a job (**Workflows › Jobs**); widget values become job parameters.
- **Dora** — AI assistant: explain, optimise or generate code for the active cell.
- **≡** table of contents · **{ }** variable explorer (try it now: `sales`, `pdf`, `result`…) · **⟲** version history · **⌨** shortcuts (`Shift+Enter` run, `B` add below, `D D` delete…).
- In a **Git folder**, the **⎇ branch** button opens Git: branches, diffs, commit & push.
- **Ask AI** (top bar, `Ctrl+J`) — a ChatGPT-style chat that can see this page.

In [ ]:
# Variables you can inspect in the { } panel:
summary = {"rows": sales.count(), "regions": [r[0] for r in sales.select("region").distinct().collect()],
           "format": TABLE_FORMAT, "widget_region": region}
summary

## 14. Errors and **Fix with Dora**
The next cell **fails on purpose** (a typo in a column name). Click **Fix with Dora** in the error to get the corrected code — or **Go to line**. It is the last cell, so *Run all* has already done everything above.

*Clean up afterwards (optional):* `spark.sql("DROP DATABASE demo CASCADE")` and `dbutils.fs.rm("tmp/feature_tour", recurse=True)`.

In [ ]:
sales.groupBy("regoin").agg(F.sum("net_sales")).show()